# 7.5 Deep Learning methods for object detection

In the previous notebooks we have classified objects from a set of features that describe them (shape, color, measurements, etc.), using classical **Machine Learning** (ML) methods and, in notebook 7.4, a first neural network: the Multilayer Perceptron. In all of them, the object had already been located (e.g. segmented), and the task was only to decide its category.

**Object detection** goes one step further: given an image, it must find **where** the objects are, enclosing each one in a **bounding box**, and **which** category each one belongs to. Nowadays, this task is dominated by **Deep Learning** (DL) models: neural networks with many layers and millions of parameters that, instead of using handcrafted features, learn them directly from the images.

The choice of ML or DL techniques strongly depends on the application at hand. DL models need large amounts of data and computational resources to be trained, and they work as black boxes that are hard to explain. Luckily, there are many models already trained with huge datasets that we can use directly, or adapt to our problem with much less data and computation (**transfer learning**).

In this notebook we will play with **YOLO**, one of the most popular families of object detectors:

- how YOLO works and how to load a pretrained model (<a href="#751">section 7.5.1</a>),
- which categories it can detect (<a href="#752">section 7.5.2</a>),
- doing inference with it (<a href="#753">section 7.5.3</a>),
- fine-tuning it to detect new categories (<a href="#754">section 7.5.4</a>), and
- doing inference with the fine-tuned model (<a href="#755">section 7.5.5</a>).

## Problem context - Object detection for mobile robots

This time we will focus on mobile robots operating in human-centered environments, for example houses. In order to provide services, a mobile robot must be able to understand its surroundings, including the elements in its workspace. For that, object detection is a critical task, since it permits the robot to locate objects that can be useful in its duties. Fig. 1 provides some examples of objects detected in images coming from a camera mounted on a mobile robot:

<center>
    <img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/object_recognition/images/dnn/examples_from_robot_at_home.PNG" width="800"/>
    <figcaption>Fig. 1. Images processed by a DL model with annotations about the detected objects.</figcaption>
</center>$\\[5pt]$

Fusing these detections with additional information, like the robot pose within the workspace and the relative pose of the camera w.r.t. the robot, the objects can be localized in said workspace. For example, Fig. 2 shows the reconstruction of two houses where the robot has detected and placed a number of objects.

<center>
    <img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/object_recognition/images/dnn/workspace_reconstructions.jpg" width="800"/>
    <figcaption>Fig. 2. Reconstructions of real houses built by a mobile robot.</figcaption>
</center>$\\[5pt]$

## Preparing our setup

In section 7.5.4 we will train a model, which needs a **GPU**. In Google Colab, select *Runtime > Change runtime type > T4 GPU* **before running any cell**, since changing it later restarts the session and you lose your work. Without a GPU, you can still run the whole notebook: a model already trained will be loaded instead.

We will use the [Ultralytics](https://docs.ultralytics.com/) library, which implements the latest versions of YOLO and permits us to run and train them with a few lines of code. It is not preinstalled in Google Colab, so let's install it:

In [ ]:
%pip install -q ultralytics==8.4.204

Now, let's import the required libraries and modules:

In [ ]:
# In case of using Google Colab, mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    # If your notebook and images are in your Drive, change directory accordingly:
    # %cd '/content/drive/MyDrive/Colab Notebooks/uma_cv_2027/Chapter 07. Object Recognition'
except ImportError:
    pass

import zlib
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (15.0, 8.0)
import torch
from ultralytics import YOLO, settings
from ultralytics.data.utils import check_det_dataset

# In Colab, keep the downloaded datasets and the training results out of Google Drive (they are large)
try:
    import google.colab
    import ultralytics.data.utils
    settings.update({'datasets_dir': '/content/datasets', 'runs_dir': '/content/runs'})
    ultralytics.data.utils.DATASETS_DIR = Path('/content/datasets')  # it was read when importing ultralytics
except ImportError:
    pass
settings.update({'sync': False})  # don't send usage analytics to Ultralytics

images_path = './images/yolo/'
print('GPU available:', torch.cuda.is_available())

## 7.5.1 The YOLO detector <a id="751"></a>

**YOLO** (*You Only Look Once*) was presented by Joseph Redmon et al. in 2016<sup>[[1]](#ref1)</sup>. Previous detectors first proposed a number of candidate regions of the image, and then classified each one with a CNN, that is, they looked at the image many times. YOLO, instead, looks at the image **only once**: a single pass of the network predicts all the bounding boxes and their categories at the same time. This makes it fast enough for real-time applications such as robotics, and since then many versions have been released, improving its accuracy and speed. We will use **YOLO26**, released by Ultralytics in 2026.

> 🔍 <font color="#0072B2"><b>AI Exploration:</b> One-stage and two-stage detectors</font>  
> **Suggested prompt**  
> *Explain the difference between two-stage object detectors (e.g. Faster R-CNN) and one-stage detectors (e.g. YOLO). Why are one-stage detectors usually preferred in real-time applications such as mobile robotics? Explain it at a beginner level.*

The idea is shown in Fig. 3: the image is divided into an $S\times S$ grid, and each cell is in charge of the objects whose **center** falls inside it. For that, each cell predicts some bounding boxes, given by their center, width and height (so a box can be much larger than its cell), each one with a **confidence** (how likely it is that the box contains an object, and how well it fits it), and a **probability per category**. Both are combined to get the final detections.

<center>
    <img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/object_recognition/images/yolo_grid_model.png" width="700"/>
    <figcaption>Fig. 3. YOLO divides the image into an S×S grid, and each cell predicts bounding boxes with their confidence and the class probabilities, which are combined into the final detections (source: YOLO paper).</figcaption>
</center>$\\[5pt]$

Modern versions, like YOLO26, keep this idea, and organize their network in three parts:

- a **backbone**, a CNN that extracts feature maps from the image,
- a **neck**, which combines feature maps of different scales, and
- a **head**, which makes the predictions with three grids of different resolution ($80\times80$, $40\times40$ and $20\times20$ cells for a $640\times640$ image), specialized in small, medium and large objects, respectively. Their boxes can have any size.

The training of these models is highly time and resource consuming, typically demanding a cluster of GPUs and huge datasets. Luckily for us, Ultralytics provides models already trained, in several sizes: from `n` (*nano*) to `x` (*extra large*). The larger the model, the more accurate, but also the slower. We will use the nano one, whose weights are downloaded the first time:

In [ ]:
# Load YOLO26 nano, pretrained with the COCO dataset
model = YOLO('yolo26n.pt')
model.info()

The network has only about 2.4 million parameters, so it can run in real time even without a GPU. Its layers are stored in `model.model.model`. Let's print their types to identify its three parts: the backbone goes from layer 0 to layer 10, the neck from layer 11 to 22 (notice the `Upsample` and `Concat` layers, which combine maps of different scales), and the last layer (`Detect`) is the head.

In [ ]:
for i, layer in enumerate(model.model.model):
    print(i, type(layer).__name__)

> 🔍 <font color="#0072B2"><b>AI Exploration:</b> The layers of YOLO26</font>  
> **Suggested prompt**  
> *The YOLO26 network of the Ultralytics library is built with layers of these types: `Conv`, `C3k2`, `SPPF`, `C2PSA`, `Upsample`, `Concat` and `Detect`. Explain at a beginner level what each type of layer does, and which role it plays in the backbone, the neck or the head of the network.*

## 7.5.2 A closed set of categories <a id="752"></a>

The categories that a detector can recognize depend on the dataset it was trained with. The model we have loaded was trained with [Microsoft COCO](https://cocodataset.org/#home)<sup>[[2]](#ref2)</sup>, the most used dataset for object detection, which considers 80 categories (see Fig. 4):

<center>
    <img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/object_recognition/images/dnn/COCO_categories.PNG" width="800"/>
    <figcaption>Fig. 4. Object categories considered in the COCO dataset (source: COCO webpage).</figcaption>
</center>$\\[5pt]$

The names of the categories are stored in `model.names`, a dictionary that maps the index of each category (the number returned by the network) to its name:

In [ ]:
print(type(model.names), len(model.names))
print(model.names[0], model.names[1], model.names[2])

### **<span style="color:green"><b><i>ASSIGNMENT 1: Exploring categories</i></b></span>**

Let's explore a bit the loaded categories. You are tasked to:

- print the categories from index 40 to index 60, with the format shown in the expected output, and
- check which of the objects in the `home_objects` list (the ones our robot should detect) are COCO categories.

In [ ]:
# Print the categories from index 40 to 60
for index in range(None, None):
    print('[', index, '] ', None)

# Check which of these objects are COCO categories
home_objects = ['bed', 'sofa', 'chair', 'table', 'lamp', 'tv', 'laptop',
                'wardrobe', 'window', 'door', 'potted plant', 'photo frame']
coco_categories = list(None)

print('\nIn COCO:', None)
print('Not in COCO:', None)

<font color='blue'>**Expected output:**  </font>

```
[ 40 ]  wine glass
[ 41 ]  cup
[ 42 ]  fork
[ 43 ]  knife
[ 44 ]  spoon
[ 45 ]  bowl
[ 46 ]  banana
[ 47 ]  apple
[ 48 ]  sandwich
[ 49 ]  orange
[ 50 ]  broccoli
[ 51 ]  carrot
[ 52 ]  hot dog
[ 53 ]  pizza
[ 54 ]  donut
[ 55 ]  cake
[ 56 ]  chair
[ 57 ]  couch
[ 58 ]  potted plant
[ 59 ]  bed
[ 60 ]  dining table

In COCO: ['bed', 'chair', 'tv', 'laptop', 'potted plant']
Not in COCO: ['sofa', 'table', 'lamp', 'wardrobe', 'window', 'door', 'photo frame']
```

Notice that `sofa` and `table` are in COCO, but with other names: `couch` and `dining table`. The rest of the objects in the second list are not, so the model **cannot detect them**: its head has exactly one score per COCO category. That is, the model works with a **closed set** of categories, and an object of any other category is either ignored or confused with the most similar known one.

> 🔍 <font color="#0072B2"><b>AI Exploration:</b> Open-vocabulary object detectors</font>  
> **Suggested prompt**  
> *The YOLO models trained with COCO can only detect 80 categories. What are open-vocabulary object detectors (e.g. YOLO-World or YOLOE), how can they detect categories given as text, and what are their limitations compared with a detector trained for a closed set of categories?*

## 7.5.3 Doing inference with the pretrained model <a id="753"></a>

**Inference** means feeding the network with an image and getting the results of its processing. With Ultralytics, it is done by calling the `predict()` method of the model with the path of the image (or a list of them). Since the model is already trained, it works well on a CPU, taking a fraction of a second per image.

`predict()` returns a list with one `Results` object per image, whose `boxes` attribute stores the detections:

- `boxes.xyxy`: the bounding boxes, as an $N\times4$ array with the coordinates of the upper left corner $[x_{ul},y_{ul}]$ and the bottom right corner $[x_{br},y_{br}]$ of each box, in pixels,
- `boxes.conf`: the confidence (score) of each detection, ranging from 0 to 1, and
- `boxes.cls`: the index of the category of each detection.

These attributes are PyTorch tensors, which we can convert into NumPy arrays with `.cpu().numpy()`. Let's run the model on an image of a living room:

In [ ]:
# Do inference with an image
result = model.predict(images_path + 'living_room.jpg', verbose=False)[0]

boxes = result.boxes.xyxy.cpu().numpy()
confidences = result.boxes.conf.cpu().numpy()
labels = [model.names[int(c)] for c in result.boxes.cls]

print('Number of detections:', len(boxes))
print('First detection:', labels[0], confidences[0], boxes[0])

<font color='blue'>**Expected output:**  </font>

```
Number of detections: 6
First detection: tv 0.80456823 [     475.61      447.83      569.04       540.9]
```

### **<span style="color:green"><b><i>ASSIGNMENT 2: Showing the results</i></b></span>**

**Complete the `draw_detections()` function**, which draws the detections over the image. For that:

- show only the detections with a confidence higher than `threshold`,
- draw the rectangles representing the bounding boxes with [`cv2.rectangle()`](https://docs.opencv.org/4.x/d6/d6e/group__imgproc__draw.html#ga07d2f74cadcf8e305e810ce8eed13bc9), using the color that the provided `category_color()` function gives to each category, and
- write over each box a text with its category and confidence, e.g. `chair (0.43)`, with [`cv2.putText()`](https://docs.opencv.org/4.x/d6/d6e/group__imgproc__draw.html#ga5126f47f883d730f633d74f07456c576).

Then, try different thresholds to see how they affect the detections, and check that with a threshold of 0.4 you get a result similar to the expected output below.

> 🛠️ <font color="#0B6B3A"><b>AI Coding Support:</b> Drawing boxes and texts with OpenCV</font>  
> **Suggested prompt**  
> *I need to draw bounding boxes and labels over an image with OpenCV's `cv2.rectangle()` and `cv2.putText()`, given the box corners as float NumPy arrays. Do not write the full solution. Explain the arguments of both functions, why the coordinates must be converted to integers, and how to place the text inside the top part of the box.*

In [ ]:
def category_color(name):
    """ Return an RGB color for a category, always the same for the same name. """
    rng = np.random.default_rng(zlib.crc32(name.encode()))
    return tuple(int(c) for c in rng.integers(0, 200, 3))

def draw_detections(image, boxes, labels, confidences, threshold=0.4):
    """ Draw the detections whose confidence is higher than a threshold.

        Args:
            image: RGB image
            boxes: N x 4 array with the [x_ul, y_ul, x_br, y_br] corners of the boxes, in pixels
            labels: list with the N category names
            confidences: array with the N confidences
            threshold: minimum confidence of the detections to draw

        Returns:
            image_copy: copy of the image with the detections drawn
    """
    image_copy = image.copy()

    for box, label, confidence in zip(boxes, labels, confidences):
        # Draw only the detections above the threshold
        if None:
            color = category_color(label)
            # Get the corners of the box
            x_ul, y_ul, x_br, y_br = None
            # Draw the bounding box
            cv2.rectangle(None)
            # Write the category and the confidence
            text = None
            cv2.putText(None)

    return image_copy

image = cv2.cvtColor(cv2.imread(images_path + 'living_room.jpg'), cv2.COLOR_BGR2RGB)
plt.imshow(draw_detections(image, boxes, labels, confidences, threshold=None))
plt.axis('off');

<font color='blue'>**Expected output:**  </font>

<img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/object_recognition/images/yolo_example_of_results.jpg" width="800"/>

Let's put the inference code in a function that detects the objects of an image, and use it with `draw_detections()` on two more images: a kitchen, and an image taken by a mobile robot.

In [ ]:
def detect(model, image_file, threshold=0.4, classes=None):
    """ Detect the objects of an image with a YOLO model.

        Args:
            model: YOLO model
            image_file: path of the image
            threshold: minimum confidence of the detections
            classes: indices of the categories to detect (None for all of them)

        Returns:
            boxes, labels, confidences: the detections, as in draw_detections()
    """
    result = model.predict(image_file, conf=threshold, classes=classes, verbose=False)[0]
    boxes = result.boxes.xyxy.cpu().numpy()
    confidences = result.boxes.conf.cpu().numpy()
    labels = [model.names[int(c)] for c in result.boxes.cls]
    return boxes, labels, confidences

test_images = ['living_room.jpg', 'kitchen.jpg', 'robot_kitchen.jpg']

plt.figure(figsize=(18, 5))
for i, file in enumerate(test_images):
    image = cv2.cvtColor(cv2.imread(images_path + file), cv2.COLOR_BGR2RGB)
    plt.subplot(1, 3, i + 1)
    plt.imshow(draw_detections(image, *detect(model, images_path + file)))
    plt.title(file)
    plt.axis('off')

<font color='blue'>**Expected output:**  </font>

<img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/object_recognition/images/yolo_expected_detections.jpg" width="1100"/>

### <font color="blue"><b><i>Thinking about it (1)</i></b></font>

Take a look at the detections of the three images and **answer these questions:**

- Which objects of the living room image are not detected? Why?

    <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>

- What do you think a detector trained with COCO would output for an object of a category that is not in COCO, e.g. a door?

    <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>

- How does the threshold affect the detections? What happens with a very low and a very high threshold?

    <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>

## 7.5.4 Fine-tuning YOLO to detect new categories <a id="754"></a>

To teach the detector new categories, we don't need to train it from scratch: we can reuse the features that it learned from COCO, and adapt it to our problem with a much smaller dataset. There are two ways of doing this **transfer learning** (see Fig. 5):

- **Fixed feature extraction**: the pretrained backbone is frozen, and only a new head is trained.
- **Fine-tuning**: the new head and some or all the layers of the backbone are trained.

<center>
    <img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/object_recognition/images/transfer_learning.png" width="1100"/>
    <figcaption>Fig. 5. Fixed feature extraction and fine-tuning, the two ways of reusing a pretrained CNN for a new task (source: theory slides).</figcaption>
</center>$\\[5pt]$

In our case, the head must be replaced, since its number of outputs depends on the number of categories, and we will **freeze the backbone** (layers 0 to 10), training the neck and the head. This reduces the training time and the risk of overfitting with a small dataset.

### The HomeObjects-3K dataset

We will use [HomeObjects-3K](https://docs.ultralytics.com/datasets/detect/homeobjects-3k/), a dataset with 2285 training images and 404 validation images of indoor scenes, annotated with 12 categories of household objects: the ones of the `home_objects` list of Assignment 1, including those that are not in COCO, like `lamp`, `window` or `door` (see Fig. 6).

<center>
    <img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/object_recognition/images/homeobjects_annotation_example.jpg" width="700"/>
    <figcaption>Fig. 6. An image from HomeObjects-3K with the annotated bounding boxes and categories of its objects.</figcaption>
</center>$\\[5pt]$

Ultralytics describes each dataset with a YAML file containing its paths, the names of its categories and its download URL. Let's download the dataset (about 400 MB) and check its information:

In [ ]:
# Download the dataset (only the first time) and get its information
data_info = check_det_dataset('HomeObjects-3K.yaml')

print('Path:', data_info['path'])
print('Categories:', data_info['names'])

Its images are stored in the `images/` folder, and their annotations in the `labels/` one, with a text file per image. Each line of these files describes an object with the **YOLO format**: `category x_center y_center width height`, that is, the index of its category and its bounding box, given by its center and size, normalized by the image size (from 0 to 1).

### Training

The `train()` method of the model fine-tunes it with a dataset. Its most relevant parameters are:

- `data`: the YAML file of the dataset.
- `epochs`: the number of passes over the training data.
- `imgsz`: the size the images are resized to, e.g. 640 for $640\times640$ images.
- `fraction`: the fraction of the training images to use. Using a part of them reduces the training time.
- `freeze`: the number of initial layers to freeze, e.g. 11 to freeze the backbone.
- `batch`: the number of images processed at the same time in each training step.

During training, Ultralytics prints the evolution of the **losses** (the box loss measures how well the boxes fit the objects, and the classification loss how well their categories are predicted) and, after each epoch, the performance on the validation images.

### **<span style="color:green"><b><i>ASSIGNMENT 3: Fine-tuning the model</i></b></span>**

**Complete the following code** to fine-tune YOLO26 nano with HomeObjects-3K. To check that your results are right, train it for 10 epochs with images of size 640, half of the training images, the backbone frozen, and a batch of 16 images. With a GPU, training takes about 6 minutes. Without it, the code loads a model already fine-tuned with these parameters.

> 🛠️ <font color="#0B6B3A"><b>AI Coding Support:</b> Training parameters of YOLO</font>  
> **Suggested prompt**  
> *I'm fine-tuning a YOLO detector with the Ultralytics library, and I already know how to use the parameters `data`, `epochs`, `imgsz`, `fraction`, `freeze` and `batch` of `model.train()`. Do not write the full solution. Which other parameters of `model.train()` are worth knowing (e.g. for data augmentation or early stopping)? Explain what each one controls and when it is worth changing its default value.*

In [ ]:
if torch.cuda.is_available():
    # Start from the model pretrained with COCO
    model_ft = YOLO('yolo26n.pt')
    model_ft.train(data=None, epochs=None, imgsz=None, fraction=None, freeze=None, batch=None,
                   name='homeobjects', exist_ok=True)
    # After training, model_ft contains the weights of the best epoch
    results_dir = model_ft.trainer.save_dir
else:
    print('No GPU available: loading a model fine-tuned with this same code')
    model_ft = YOLO('./data/yolo26n_homeobjects.pt')
    results_dir = None

Ultralytics saves the results of the training in a folder: the weights of the last and best epochs (`weights/last.pt` and `weights/best.pt`), and some plots, like the evolution of the losses and the metrics in `results.png`:

In [ ]:
if results_dir is not None:
    print('Results saved in', results_dir)
    plt.imshow(plt.imread(results_dir / 'results.png'))
    plt.axis('off');

### <font color="blue"><b><i>Thinking about it (2)</i></b></font>

Now that you have fine-tuned the model, **answer these questions:**

- Why is it useful to freeze the backbone? What would you expect if we trained the whole network, or for more epochs?

    <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>

- The fine-tuned model has a head with 12 outputs. What has happened with the knowledge of the other COCO categories?

    <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>

### **<span style="color:green"><b><i>OPTIONAL</i></b></span>**

How good is the fine-tuned model? Evaluate it with the validation images of HomeObjects-3K using its [`val()`](https://docs.ultralytics.com/modes/val/) method, and find out what the metrics it prints mean (e.g. mAP50). Which categories are detected best and worst? Why do you think some of them are harder to detect?

### **<span style="color:green"><b><i>END OF OPTIONAL PART</i></b></span>**

## 7.5.5 Doing inference with the fine-tuned model <a id="755"></a>

Let's compare the detections of both models in our test images:

In [ ]:
plt.figure(figsize=(18, 10))
for i, file in enumerate(test_images):
    image = cv2.cvtColor(cv2.imread(images_path + file), cv2.COLOR_BGR2RGB)
    plt.subplot(2, 3, i + 1)
    plt.imshow(draw_detections(image, *detect(model, images_path + file)))
    plt.title('Pretrained: ' + file)
    plt.axis('off')
    plt.subplot(2, 3, i + 4)
    plt.imshow(draw_detections(image, *detect(model_ft, images_path + file)))
    plt.title('Fine-tuned: ' + file)
    plt.axis('off')

<font color='blue'>**Expected output:**  </font>

With the provided fine-tuned model (your results may differ slightly if you trained your own):

<img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/object_recognition/images/yolo_expected_comparison.jpg" width="1100"/>

The fine-tuned model detects the new categories, like doors, windows or lamps, but it has lost all the COCO categories that are not in HomeObjects-3K, like the microwave and the refrigerator of the kitchen: it works with **a new closed set** of 12 categories. To keep the COCO categories, the training dataset should include all of them, annotated in every image.

A simple alternative is to **combine both models**: we keep all the detections of the pretrained model, and add those of the fine-tuned model that belong to the new categories. The `classes` parameter of our `detect()` function restricts the detections to some categories.

### **<span style="color:green"><b><i>ASSIGNMENT 4: Combining both models</i></b></span>**

**Complete the following code** to:

- get the indices of the categories of the fine-tuned model that are not in COCO (`new_categories`), keeping in mind that `sofa` and `table` are in COCO with other names, and
- combine the detections of both models in the kitchen image, and draw them.

In [ ]:
# Indices of the fine-tuned categories that are not in COCO (sofa and table are, as couch and dining table)
coco_categories = list(model.names.values()) + ['sofa', 'table']
new_categories = None
print('New categories:', [model_ft.names[i] for i in new_categories])

# Detect with both models
file = images_path + 'kitchen.jpg'
boxes_coco, labels_coco, conf_coco = detect(None, file)
boxes_new, labels_new, conf_new = detect(None, file, classes=None)

# Combine the detections
boxes = np.vstack([None, None])
labels = None
confidences = np.concatenate([None, None])

image = cv2.cvtColor(cv2.imread(file), cv2.COLOR_BGR2RGB)
plt.imshow(draw_detections(image, boxes, labels, confidences))
plt.axis('off');

<font color='blue'>**Expected output:**  </font>

```
New categories: ['lamp', 'wardrobe', 'window', 'door', 'photo frame']
```

<img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/object_recognition/images/yolo_combined_detections.jpg" width="800"/>

### <font color="blue"><b><i>Thinking about it (3)</i></b></font>

Now that you have compared and combined both models, **answer these questions:**

- Which are the advantages and disadvantages of combining two models instead of using only one?

    <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>

- Could the combined detector give two boxes for the same object? In which cases?

    <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>

> 🧭 <font color="#B8860B"><b>AI Reflection:</b> Before and after fine-tuning</font>  
> **Suggested prompt**  
> *I fine-tuned a YOLO detector pretrained with COCO using the HomeObjects-3K dataset. The fine-tuned model detects doors, windows and lamps, but it no longer detects COCO categories like microwaves or refrigerators. My explanation of why this happens is: [your explanation]. Guide me, without giving me the answers, to check my reasoning and to think about other ways to get a single model that detects all the categories.*

✍️ **Your notes / key takeaways:**

<p style="margin: 4px 0px 6px 5px; color:blue"><i>Write your conclusions here after discussing with your assistant!</i></p>

With this type of models onboard, a mobile robot would be able to detect the objects in its surroundings, including the ones needed for its specific tasks, and use this information during its operation. Awesome!

### **<span style="color:green"><b><i>OPTIONAL</i></b></span>**

Now that you know how to use and fine-tune YOLO, you can challenge the models with your own images, or fine-tune them with other datasets supported by Ultralytics, like [African Wildlife](https://docs.ultralytics.com/datasets/detect/african-wildlife/) or [Medical Pills](https://docs.ultralytics.com/datasets/detect/medical-pills/). You can also try other tasks that YOLO supports, like instance segmentation or pose estimation.

### **<span style="color:green"><b><i>END OF OPTIONAL PART</i></b></span>**

## Conclusion

Cool! You have experienced the trendy Deep Learning world a bit.

In this notebook you have learned:

- how YOLO detects objects in a single pass, with a backbone, a neck and a head,
- that a detector can only recognize the closed set of categories it was trained with,
- how to do inference with a pretrained detector and show its results,
- how to fine-tune a detector with a new dataset to detect new categories, and
- how to combine two detectors to keep both the old and the new categories.

## <font color="orange"><b><i>AI Appendix</i></b></font>

- **Model/tool used:**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>

- **What I used AI for:**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Briefly explain how you used AI in this notebook: whether you followed the proposed AI interactions, adapted them, skipped some of them, or used AI for additional purposes such as debugging, clarifying concepts, improving prompts or checking your reasoning.</i></p>

- **Most and least useful proposed AI interactions:**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>By proposed AI interactions, we mean the AI prompts explicitly included in the notebook. Choose one that helped you the most and one that helped you the least. Briefly explain why.</i></p>

- **Most useful personal AI interaction:**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>By personal AI interaction, we mean any additional prompt or follow-up question that you wrote yourself, beyond the prompts proposed in the notebook. Paste or summarize one, if any, and explain why it helped.</i></p>

- **What I verified, adapted or decided myself:**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Briefly explain what you checked, corrected, adapted or decided before using the AI output in your final work.</i></p>

## References

<a name="ref1">[1]</a>: REDMON, Joseph, et al. [You Only Look Once: Unified, Real-Time Object Detection](https://arxiv.org/abs/1506.02640). In IEEE Conference on Computer Vision and Pattern Recognition (CVPR), 2016. p. 779-788.

<a name="ref2">[2]</a>: LIN, Tsung-Yi, et al. [Microsoft COCO: Common Objects in Context](https://arxiv.org/abs/1405.0312). In European Conference on Computer Vision (ECCV). Springer, Cham, 2014. p. 740-755.